<div style="background:#E9FFF6; color:#440404; padding:8px; border-radius: 4px; text-align: center; font-weight: 500;">IFN619 - Data Analytics for Strategic Decision Makers</div>

# IFN619 :: C1-Semi/unstructured Analytics Tutorial Exercises (Part B)

For this tutorial, we will continue the same scenario and process as Part A of the tutorial notebook (B3). Recall that we used "Brisbane 2032 Olympics" as a suggested topic and completed the following process:
1. Use the Guardian API to undertake your own search and obtain a json file of documents
2. Create a TF/IDF document-term matrix for your documents

For this tutorial session, we are going to extend our process to include topic modelling.

In [ ]:
# Import the necessary libraries
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF
import pandas as pd
import json
import random

### Question

What is the question you investigated in B3, or what question would you like to investigate: 

**Question**: [Write your question here and explain why it's significant.] 

### B3 recap and set up for new analyses

We need to load the articles we previously collected, and extend our dataframe to include `lda` and `nmf` for comparison.

To save time, we will reuse code from B3 tutorial directly. For explanations and notes, refer back to B3 tutorial notebook. 

#### Load data and find relevant articles

In [ ]:
# Load the data - articles saved from B3 tutorial
file_path = "data/"
file_name = "???.json"

with open(f"{file_path}{file_name}",'r', encoding='utf-8') as fp:
    articles = json.load(fp)

print(f"Loaded {len(articles)} articles from {file_name}")

In [ ]:
# process articles to find relevant articles for the topic
## You might need to customise this part if you use a different topic 

# e.g., remove articles contains 'as it happened' - why?
## get a list of article titles 
titles = list(articles.keys())
## create an empty list to store filtered titles
filtered_titles = []

for title in titles:
    if "as it happens" not in title:
        filtered_titles.append(title)
        
# e.g., include titles that contain 'Brisbane' and 'Olympic' - why?
filtered_titles_2 = [title for title in filtered_titles if 'Brisbane' in title and 'Olympic' in title]

# Filter the JSON data to only include these titles
articles_filtered = {title: content for title, content in articles.items() if title in filtered_titles_2}
len(articles_filtered)

#### Create a top10 terms dataframe

Using the index from the documents, create a dataframe that can hold the top10 terms for each document. This time we will include `lda` and `nmf` in the dataframe.

In [ ]:
# Create a dataframe to hold top terms for each analysis type
terms_df = pd.DataFrame(index=???.keys(),columns=['count', 'tfidf', 'lda', 'nmf'])
terms_df

#### Previous analyses for term frequency and TFIDF

In [ ]:
# term frequency
# Set parameters appropriate to your data
count_vectorizer = CountVectorizer(max_df=???, min_df=???, max_features=???, stop_words="english")
count_dt_matrix = count_vectorizer.fit_transform(articles_filtered.values())
# Get the terms identified during the vectorization process
feature_names = count_vectorizer.get_feature_names_out()
# Create a new dataframe with the matrix - use titles for the index and terms for the columns
count_df = pd.DataFrame(count_dt_matrix.toarray(), index = terms_df.index, columns=feature_names)
#For each doc, get the 10 columns with the largest counts
for idx in terms_df.index:
    counts = dict(count_df.loc[idx].sort_values(ascending=False).head(10))
    #print(counts)
    terms_df.at[idx,'count'] = list(counts.keys()) # Just the list of terms

# tfidf
# Set parameters appropriate to your data
tfidf_vectorizer = TfidfVectorizer(max_df=???, min_df=???, max_features=???, stop_words="english")
# Get the document vectors
tfidf_dt_matrix = tfidf_vectorizer.fit_transform(articles_filtered.values())
# list of feature names
feature_names = tfidf_vectorizer.get_feature_names_out()
# create a df to combine matrix with feature names
tfidf_df = pd.DataFrame(tfidf_dt_matrix.toarray(), index=articles_filtered.keys(), columns=feature_names)
for idx in terms_df.index:
    tfidf = dict(tfidf_df.loc[idx].sort_values(ascending=False).head(10))
    terms_df.at[idx,'tfidf'] = list(tfidf.keys())

terms_df

### Topic modelling with Latent Dirichlet Allocation (LDA)
We will follow the same process from the lecture to extract top 10 terms using LDA. It's important to refer to the lecture notebook to understand how topic modelling work. Explore different parameters.

In [ ]:
# Set number of topics
num_topics = ???
# Set max number of iteractions
max_iterations = ???

# Create the model
lda_model = LatentDirichletAllocation(n_components=???, max_iter=???,learning_method='online')

# Fit the model to the data, and use the model to transform the data (do the decomposition)
doc_topic_matrix = lda_model.fit_transform(???)

# Obtain the topics
topic_term_matrix = lda_model.components_

#### View the topics

In [ ]:
# Get the topics and their terms
lda_topic_dict = {}
for index, topic in enumerate(???):
    zipped = zip(feature_names, topic)
    top_terms=dict(sorted(zipped, key = lambda t: t[1], reverse=True)[:10])
    #print(top_terms)
    top_terms_list= {key : round(top_terms[key], 4) for key in top_terms.keys()}
    lda_topic_dict[f"topic_{index}"] = top_terms_list

# Print the topics with their terms    
for k,v in lda_topic_dict.items():
    print(k)
    print(v)
    print()

##### Discussions

1. What kinds of 'themes' can we infer from each topic? Can we get a sense what most of the articles focus on? 
- [Jot down your notes here]

2. How might we be able to use insights from the topics to address your question?
- [Jot down your notes here]

#### Update the terms matrix

> Note: the code below is slightly different from the lecture, as in our previous practice we set the article names as the dataframe index instead of using the default index.

In [ ]:
for article_name,topic in zip(terms_df.index, ???):
    topic_num = topic.argmax() # which item (the location) has the max value
    top_topic = lda_topic_dict[f"topic_{topic_num}"]
    terms_df.at[article_name,'lda'] = list(top_topic.keys())

terms_df

### Topic modelling with Non-negative Matrix Factorisation (NMF)


[NMF](https://en.wikipedia.org/wiki/Latent_Dirichlet_allocation) is a different algorithm for obtaining *topics* (a list of terms) from a document-term matrix. It also factorises the document-term matrix into 2 factor matrices: document-topic and topic-term.

In [ ]:
# Set the number of topics
num_topics = ???

# Create the model
nmf_model = NMF(n_components=???, init='random', beta_loss='frobenius')

# Fit the model to the data and use it to transform the data
doc_topic_nmf = nmf_model.fit_transform(???)

topic_term_nmf = nmf_model.components_

In [ ]:
# Get the topics and their terms
nmf_topic_dict = {}
for index, topic in enumerate(???):
    zipped = zip(feature_names, topic)
    top_terms=dict(sorted(zipped, key = lambda t: t[1], reverse=True)[:10])
    #print(top_terms)
    top_terms_list= {key : round(top_terms[key], 4) for key in top_terms.keys()}
    nmf_topic_dict[f"topic_{index}"] = top_terms_list

# Print the topics with their terms    
for k,v in nmf_topic_dict.items():
    print(k)
    print(v)
    print()

##### Discussions
1. What kinds of 'themes' can we infer from each topic? Can we get a sense what most of the articles focus on? Any similarities/differences from the LDA results?
- [Jot down your notes here]

2. How might we be able to use insights from the topics to address your question?
- [Jot down your notes here]

#### Update the terms matrix

In [ ]:
for article_name,topic in zip(terms_df.index, ???):
    topic_num = topic.argmax()
    top_topic = nmf_topic_dict[f"topic_{topic_num}"]
    terms_df.at[article_name,'nmf'] = list(top_topic.keys())

terms_df

### Check against articles

In [ ]:
# Sample 5 random articles
samples = random.sample(range(0,len(terms_df)),5)

for sample in samples:
    doc = terms_df.iloc[sample]
    print(f"[{sample}] {doc.index}")
    print("\t>> Counts:\t",doc['count'])
    print("\t>> TFIDF:\t",doc['tfidf'])
    print("\t>> LDA:\t\t",doc['lda'])
    print("\t>> NMF:\t\t",doc['nmf'])
    print()

**What can you find from the comparison? Is it helpful to address your question? If not, what can you get out of it and use it for further investigation?**

[Write your thoughts here]

In [ ]:
# Continue your investigations - add more cells


## Refine your analysis

Once you have worked through the process. Try tweaking the parameters to obtain better results for your data.

#### Advanced

You may obtain better results by doing the following:

- Creating smaller documents (e.g. article paragraphs)
- Pre-processing the text by Stemming or Lemmatizing, and by removing additional stop words.
- ???